**The following cell comes from Kaggle Notebooks and is used to automatically download all relevant data inputs**

In [17]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/make-data-count-finding-data-references/sample_submission.csv
/kaggle/input/make-data-count-finding-data-references/train_labels.csv
/kaggle/input/make-data-count-finding-data-references/test/XML/10.1002_ece3.5260.xml
/kaggle/input/make-data-count-finding-data-references/test/XML/10.1002_chem.201902131.xml
/kaggle/input/make-data-count-finding-data-references/test/XML/10.1002_ece3.3985.xml
/kaggle/input/make-data-count-finding-data-references/test/XML/10.1002_ejoc.202000916.xml
/kaggle/input/make-data-count-finding-data-references/test/XML/10.1002_ece3.6144.xml
/kaggle/input/make-data-count-finding-data-references/test/XML/10.1002_2017jc013030.xml
/kaggle/input/make-data-count-finding-data-references/test/XML/10.1002_anie.201916483.xml
/kaggle/input/make-data-count-finding-data-references/test/XML/10.1002_chem.202001412.xml
/kaggle/input/make-data-count-finding-data-references/test/XML/10.1002_chem.202003167.xml
/kaggle/input/make-data-count-finding-data-references/test/X


# MDC Submission Notebook with PDF + XML Fallback

This notebook extracts dataset citations from scientific articles using both PDF and XML formats, classifies them as **Primary** or **Secondary**, and prepares a `submission.csv` for the Make Data Count Kaggle competition.

In [3]:
#pip install pdfplumber
!pip install langchain_openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.8/442.8 kB 13.3 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.66
    Uninstalling langchain-core-0.3.66:
      Successfully uninstalled langchain-core-0.3.66


In [5]:

import os, re
import pandas as pd
import pdfplumber
import xml.etree.ElementTree as ET
from tqdm import tqdm
from langchain_openai import ChatOpenAI
from kaggle_secrets import UserSecretsClient

print('Done!')


Done!


In [7]:

# Load OpenAI API key from Kaggle Secrets
user_secrets = UserSecretsClient()
os.environ["OPENAI_API_KEY"] = user_secrets.get_secret("OPENAI_API_KEY")

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o", temperature=0)

print('Done!')

Done!


In [ ]:

# Extract text from PDF using pdfplumber and from XML using ElementTree

def extract_text_from_pdf(pdf_path): 
    try:
        with pdfplumber.open(pdf_path) as pdf:
            return " ".join(page.extract_text() or "" for page in pdf.pages)
    except Exception as e:
        print(f"PDF extraction error: {e}")
        return ""

def extract_text_from_xml(xml_path):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        text_content = []
        for elem in root.iter():
            if elem.text:
                text_content.append(elem.text.strip())
        return " ".join(text_content)
    except Exception as e:
        print(f"XML extraction error: {e}")
        return ""

print('Done!')

Done!


In [ ]:

# Simple regex to find DOIs in the text (e.g. "10.1234/abcde" or "https://doi.org/10.1234/abcde")

def extract_dois(text): 
    doi_regex = r"(?:https?://doi\.org/)?(10\.\d{4,9}/[-._;()/:A-Z0-9]+)"
    return list(set(re.findall(doi_regex, text, flags=re.I)))

# Heuristic rules + optional GPT-4 classification to determine if a citation context refers to a Primary or Secondary dataset

def classify_citation_context(text, doi, use_gpt=True): 
    lower_text = text.lower()
    if any(kw in lower_text for kw in ["we collected", "generated our data", "new data"]):
        return "Primary"
    if any(kw in lower_text for kw in ["obtained from", "downloaded from", "existing dataset", "reused"]):
        return "Secondary"
    if use_gpt:
        try:
            prompt = f"""The following sentence refers to dataset {doi}:

            {text.strip()}

            Is this a Primary or Secondary dataset?"""

            reply = llm.invoke(prompt)
            return "Primary" if "primary" in reply.content.lower() else "Secondary"
        except Exception as e:
            print(f"OpenAI error: {e}")
            return "Unknown"
    return "Unknown"

print('Done!')

Done!


In [19]:
import os
import pandas as pd

# Set paths
pdf_dir = "/kaggle/input/make-data-count-finding-data-references/test/PDF"
xml_dir = "/kaggle/input/make-data-count-finding-data-references/test/XML"

# Get first article ID
article_ids = sorted({os.path.splitext(f)[0] for f in os.listdir(pdf_dir)} | 
                     {os.path.splitext(f)[0] for f in os.listdir(xml_dir)})
article_id = article_ids[0]
rows = []

# File paths
pdf_path = os.path.join(pdf_dir, article_id + ".pdf")
xml_path = os.path.join(xml_dir, article_id + ".xml")

# Extract text
text = extract_text_from_pdf(pdf_path)
if not text:
    text = extract_text_from_xml(xml_path)

# Process if text found
if text:
    print(f"Processing article: {article_id}")
    for doi in extract_dois(text):
        doi_url = f"https://doi.org/{doi}"
        snippet = next((s for s in text.split('.') if doi in s), text[:300])
        citation_type = classify_citation_context(snippet, doi)
        
        print(f"\n📄 Article: {article_id}")
        print(f"🔗 DOI: {doi_url}")
        print(f"🧠 Type: {citation_type}")
        print(f"📝 Snippet: {snippet.strip()[:300]}...\n")
        
        if citation_type in ["Primary", "Secondary"]:
            rows.append((article_id, doi_url, citation_type))
else:
    print(f"No text extracted from article: {article_id}")

# Save to CSV
df = pd.DataFrame(rows, columns=["article_id", "dataset_id", "type"])
df.to_csv("/kaggle/working/submission.csv", index=False)
df.head()


Processing article: 10.1002_2017jc013030

📄 Article: 10.1002_2017jc013030
🔗 DOI: https://doi.org/10.1029/2010JC006899
🧠 Type: Primary
📝 Snippet: PUBLICATIONS
Journal of Geophysical Research: Oceans
RESEARCH ARTICLE Assessing the Variability in the Relationship Between the
10.1002/2017JC013030 Particulate Backscattering Coefficient and the Chlorophyll a
Concentration From a Global Biogeochemical-Argo Database
KeyPoints:
(cid:2)Thebbp-to-Chlar...


📄 Article: 10.1002_2017jc013030
🔗 DOI: https://doi.org/10.5194/bg-7-2117-2010
🧠 Type: Primary
📝 Snippet: PUBLICATIONS
Journal of Geophysical Research: Oceans
RESEARCH ARTICLE Assessing the Variability in the Relationship Between the
10.1002/2017JC013030 Particulate Backscattering Coefficient and the Chlorophyll a
Concentration From a Global Biogeochemical-Argo Database
KeyPoints:
(cid:2)Thebbp-to-Chlar...


📄 Article: 10.1002_2017jc013030
🔗 DOI: https://doi.org/10.1002/lom3.10144
🧠 Type: Primary
📝 Snippet: PUBLICATIONS
Journal of Geophysical R

,article_id,dataset_id,type
0,10.1002_2017jc013030,https://doi.org/10.1029/2010JC006899,Primary
1,10.1002_2017jc013030,https://doi.org/10.5194/bg-7-2117-2010,Primary
2,10.1002_2017jc013030,https://doi.org/10.1002/lom3.10144,Primary
3,10.1002_2017jc013030,https://doi.org/10.1093/plankt/fbh012,Primary
4,10.1002_2017jc013030,https://doi.org/10.4319/lo.1996.41.8.,Primary
